###Optimize Threshold

In [ ]:
#Cost based threshold optimization for business impact
best_model = models[best_model_name]
y_pred_proba =  best_model.predict_proba(X_test_scaled)[:,1]

thresholds = np.arange(0.01,1.0,0.01)
cost_fp = 1
cost_fn = 10
costs=[]
metrics= []

for threshold in thresholds:
  y_pred = (y_pred_proba >= threshold).astype(int)

  tn,fp,fn,tp = confusion_matrix(y_test,y_pred).ravel()

  total_cost = (fp*cost_fp) + (fn*cost_fn)
  costs.append(total_cost)

  precision = tp/(tp+fp) if (tp+fp)>0 else 0
  recall = tp/(tp+fn) if (tp+fn) else 0
  f1 = 2 * (precision*recall) / (precision + recall) if (precision+recall) >0 else 0

  metrics.append({
      'threshold': threshold,
      'cost': total_cost,
      'precision': precision,
      'recall': recall,
      'f1': f1,
      'tp':tp,'fp':fp,'tn':tn, 'fn':fn
      })

optimal_idx = np.argmin(costs)
optimal_threshold = thresholds[optimal_idx]
optimal_metrics = metrics[optimal_idx]

print(f"Optimal threshold: {optimal_threshold:.3f}")
print(f"• Minimum cost: ${optimal_metrics['cost']:.0f}")
print(f"• Precision: {optimal_metrics['precision']:.3f}")
print(f"• Recall: {optimal_metrics['recall']:.3f}")
print(f"• F1-Score: {optimal_metrics['f1']:.3f}")
print(f"• True Positives: {optimal_metrics['tp']}")
print(f"• False Positives: {optimal_metrics['fp']}")
print(f"• False Negatives: {optimal_metrics['fn']}")

Optimal threshold: 0.700
• Minimum cost: $162
• Precision: 0.626
• Recall: 0.888
• F1-Score: 0.734
• True Positives: 87
• False Positives: 52
• False Negatives: 11


In [ ]:
risk_thresholds = {}

risk_thresholds['optimal'] = optimal_threshold
risk_thresholds['conservative'] = optimal_threshold * 0.7  # Lower threshold for higher recall
risk_thresholds['aggressive'] = optimal_threshold * 1.3   # Higher threshold for higher precision

In [ ]:
optimal_metrics

{'threshold': np.float64(0.7000000000000001),
 'cost': np.int64(162),
 'precision': np.float64(0.6258992805755396),
 'recall': np.float64(0.8877551020408163),
 'f1': np.float64(0.7341772151898734),
 'tp': np.int64(87),
 'fp': np.int64(52),
 'tn': np.int64(56812),
 'fn': np.int64(11)}

###SHAP Model Explainations

In [ ]:
#Shap explainability for audit and compliance
best_model = models[best_model_name]

sample_size = 1000
sample_indices = np.random.choice(len(X_test_scaled),
                                        min(sample_size, len(X_test_scaled)),
                                        replace=False)

X_sample = X_test_scaled[sample_indices]

In [ ]:
# Create SHAP explainer
if best_model_name in ['lightgbm', 'xgboost']:
  explainer = shap.TreeExplainer(best_model)
else:
  explainer = shap.LinearExplainer(best_model, X_train_scaled)

In [ ]:
# Calculate SHAP values
shap_values = explainer.shap_values(X_sample)

# For binary classification, get positive class SHAP values
if len(shap_values) == 2:
  shap_values = shap_values[1]

In [ ]:
# Feature importance summary
feature_importance = np.abs(shap_values).mean(axis=0)
feature_importance_df = pd.DataFrame({
    'feature': feature_names,
    'importance': feature_importance
}).sort_values('importance', ascending=False)

for i, row in feature_importance_df.head(10).iterrows():
  print(f"• {row['feature']}: {row['importance']:.4f}")

shap_values = shap_values
shap_explainer = explainer
feature_importance = feature_importance_df

• V14: 1.4311
• V4: 1.2474
• PCA_magnitude: 0.7597
• V12: 0.4299
• V10: 0.3652
• V1: 0.3609
• V3: 0.3316
• V8: 0.3180
• Time_diff: 0.3150
• V18: 0.2147


In [ ]:
feature_importance_df

,feature,importance
13,V14,1.431116
3,V4,1.247364
35,PCA_magnitude,0.759704
11,V12,0.429938
9,V10,0.365245
0,V1,0.360861
2,V3,0.331631
7,V8,0.317982
31,Time_diff,0.314973
17,V18,0.214739
